# Week 3: Model Evaluation and Final Results

**Team:** Nasiru and Jose  
**Goal:** Test saved model thoroughly and create final presentation  
**Focus:** ROC curves, confusion matrix, professional results

## 1. Load Libraries and Saved Model

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
# Load dataset and saved model
print("LOADING DATA AND SAVED MODEL")
print("="*40)

# Load data
try:
    for filepath in ['../original.csv', '../training_data.csv', 'clean_heart_failure_data.csv']:
        try:
            df = pd.read_csv(filepath)
            print(f"Dataset loaded from: {filepath}")
            break
        except FileNotFoundError:
            continue
except Exception as e:
    print(f"Error loading data: {e}")

# Prepare data (same split as training)
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Test set size: {X_test.shape[0]} patients")

# Load saved model
try:
    with open('../best_model.pkl', 'rb') as f:
        model = pickle.load(f)
    print("Saved model loaded successfully")
    print(f"Model type: {type(model).__name__}")
except FileNotFoundError:
    print("Error: best_model.pkl not found. Run week2_modeling.ipynb first.")
except Exception as e:
    print(f"Error loading model: {e}")

## 2. Comprehensive Model Evaluation

In [ ]:
# Make predictions and calculate basic metrics
print("COMPREHENSIVE MODEL EVALUATION")
print("="*40)

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
HOSPITAL_BASELINE = 0.839
improvement = accuracy - HOSPITAL_BASELINE

print(f"Model Performance:")
print(f"  Accuracy: {accuracy:.3f} ({accuracy:.1%})")
print(f"  Hospital Baseline: {HOSPITAL_BASELINE:.3f} ({HOSPITAL_BASELINE:.1%})")
print(f"  Improvement: {improvement:+.3f} ({improvement:+.1%})")
print(f"  Beats Baseline: {'YES' if accuracy > HOSPITAL_BASELINE else 'NO'}")

# Detailed classification report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Survived', 'Died']))

## 3. Confusion Matrix Analysis

In [ ]:
# Create detailed confusion matrix analysis
print("CONFUSION MATRIX ANALYSIS")
print("="*40)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix Breakdown:")
print(f"  True Negatives (Correct Survival): {tn}")
print(f"  False Positives (Wrong Death Prediction): {fp}")
print(f"  False Negatives (Missed Deaths): {fn}")
print(f"  True Positives (Correct Death Prediction): {tp}")

# Calculate clinical metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"\nClinical Metrics:")
print(f"  Sensitivity (Death Detection): {sensitivity:.3f} ({sensitivity:.1%})")
print(f"  Specificity (Survival Detection): {specificity:.3f} ({specificity:.1%})")
print(f"  Positive Predictive Value: {ppv:.3f} ({ppv:.1%})")
print(f"  Negative Predictive Value: {npv:.3f} ({npv:.1%})")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Survived', 'Died'],
            yticklabels=['Survived', 'Died'])
plt.title('Confusion Matrix - Heart Failure Prediction')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# Add percentage annotations
total = np.sum(cm)
for i in range(2):
    for j in range(2):
        plt.text(j+0.5, i+0.7, f'({cm[i,j]/total:.1%})', 
                ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

## 4. ROC Curve Analysis

In [ ]:
# Create ROC curve analysis
print("ROC CURVE ANALYSIS")
print("="*40)

if y_prob is not None:
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    
    print(f"Area Under Curve (AUC): {auc_score:.3f}")
    print(f"AUC Interpretation:")
    if auc_score >= 0.9:
        print("  Excellent discrimination (AUC ≥ 0.9)")
    elif auc_score >= 0.8:
        print("  Good discrimination (0.8 ≤ AUC < 0.9)")
    elif auc_score >= 0.7:
        print("  Fair discrimination (0.7 ≤ AUC < 0.8)")
    else:
        print("  Poor discrimination (AUC < 0.7)")
    
    # Plot ROC curve
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'Model (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], color='red', lw=1, linestyle='--', label='Random (AUC = 0.50)')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.title('ROC Curve - Heart Failure Prediction')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
else:
    print("ROC curve not available (model doesn't output probabilities)")
    auc_score = None

## 5. Feature Importance Analysis

In [ ]:
# Analyze feature importance
print("FEATURE IMPORTANCE ANALYSIS")
print("="*40)

if hasattr(model, 'feature_importances_'):
    # Get feature importance
    importances = model.feature_importances_
    
    # Create feature importance dataframe
    feature_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("Top 5 Most Important Features:")
    for i, (_, row) in enumerate(feature_df.head(5).iterrows(), 1):
        print(f"  {i}. {row['Feature']}: {row['Importance']:.3f}")
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    top_features = feature_df.head(8)
    bars = plt.barh(range(len(top_features)), top_features['Importance'][::-1])
    plt.yticks(range(len(top_features)), top_features['Feature'][::-1])
    plt.xlabel('Feature Importance')
    plt.title('Top Features for Heart Failure Prediction')
    
    # Add value labels on bars
    for i, (bar, importance) in enumerate(zip(bars, top_features['Importance'][::-1])):
        plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{importance:.3f}', va='center')
    
    plt.tight_layout()
    plt.show()
    
else:
    print(f"Feature importance not available for {type(model).__name__}")
    feature_df = None

## 6. Clinical Insights and Impact

In [ ]:
# Provide clinical insights
print("CLINICAL INSIGHTS")
print("="*40)

# Overall survival analysis
overall_survival = 1 - df['DEATH_EVENT'].mean()
print(f"Overall survival rate in dataset: {overall_survival:.1%}")

# High-risk patient analysis
if 'ejection_fraction' in df.columns:
    high_risk = df['ejection_fraction'] < 40  # Clinical threshold
    if high_risk.sum() > 0:
        high_risk_mortality = df[high_risk]['DEATH_EVENT'].mean()
        print(f"High-risk patients (EF < 40%): {high_risk.sum()} patients")
        print(f"Mortality rate in high-risk group: {high_risk_mortality:.1%}")

# Age-based analysis
if 'age' in df.columns:
    elderly = df['age'] >= 75
    if elderly.sum() > 0:
        elderly_mortality = df[elderly]['DEATH_EVENT'].mean()
        print(f"Elderly patients (≥75 years): {elderly.sum()} patients")
        print(f"Mortality rate in elderly: {elderly_mortality:.1%}")

print(f"\nClinical Impact:")
print(f"  • This model helps prioritize high-risk patients")
print(f"  • Early identification enables better resource allocation")
print(f"  • Improved prediction accuracy can save lives")
print(f"  • Model can assist in clinical decision-making")

# Model interpretation
if feature_df is not None and len(feature_df) > 0:
    top_feature = feature_df.iloc[0]['Feature']
    print(f"\nKey Finding: {top_feature} is the most predictive feature")
    if top_feature == 'ejection_fraction':
        print(f"  Clinical relevance: Low ejection fraction (<40%) indicates heart failure")
    elif top_feature == 'serum_creatinine':
        print(f"  Clinical relevance: High creatinine indicates kidney dysfunction")
    elif top_feature == 'age':
        print(f"  Clinical relevance: Advanced age increases mortality risk")

## 7. Final Results Dashboard

In [ ]:
# Create comprehensive results dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Performance vs Baseline
models = ['Hospital\nBaseline', 'Our\nModel']
scores = [HOSPITAL_BASELINE, accuracy]
colors = ['lightcoral', 'lightgreen' if accuracy > HOSPITAL_BASELINE else 'orange']

bars = axes[0,0].bar(models, scores, color=colors)
axes[0,0].set_ylabel('Accuracy')
axes[0,0].set_title('Model Performance vs Hospital Baseline')
axes[0,0].set_ylim(0.7, 1.0)

# Add value labels
for bar, score in zip(bars, scores):
    height = bar.get_height()
    axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                  f'{score:.1%}', ha='center', va='bottom', fontweight='bold')

# 2. Confusion Matrix (simplified)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1],
           xticklabels=['Survived', 'Died'],
           yticklabels=['Survived', 'Died'])
axes[0,1].set_title('Confusion Matrix')

# 3. Clinical Metrics
metrics = ['Sensitivity\n(Death Detection)', 'Specificity\n(Survival Detection)', 
          'Overall\nAccuracy']
values = [sensitivity, specificity, accuracy]

bars = axes[1,0].bar(metrics, values, color=['skyblue', 'lightgreen', 'gold'])
axes[1,0].set_ylabel('Score')
axes[1,0].set_title('Clinical Performance Metrics')
axes[1,0].set_ylim(0, 1)

for bar, value in zip(bars, values):
    height = bar.get_height()
    axes[1,0].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                  f'{value:.1%}', ha='center', va='bottom')

# 4. Feature Importance (if available)
if feature_df is not None:
    top_5 = feature_df.head(5)
    axes[1,1].barh(range(len(top_5)), top_5['Importance'][::-1])
    axes[1,1].set_yticks(range(len(top_5)))
    axes[1,1].set_yticklabels(top_5['Feature'][::-1])
    axes[1,1].set_xlabel('Importance')
    axes[1,1].set_title('Top 5 Predictive Features')
else:
    axes[1,1].text(0.5, 0.5, 'Feature importance\nnot available', 
                  ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('Feature Analysis')

plt.suptitle('Heart Failure Prediction - Final Results Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Final Summary and Project Completion

In [ ]:
# Create final project summary
print("FINAL PROJECT SUMMARY")
print("="*60)

summary = {
    'Project': 'Heart Failure Prediction',
    'Team': 'Goblins Hiding in Vents',
    'Dataset': f'{len(df)} heart failure patients from UCI',
    'Features': f'{len(X.columns)} clinical features',
    'Model Type': type(model).__name__,
    'Final Accuracy': f"{accuracy:.3f} ({accuracy:.1%})",
    'Hospital Baseline': f"{HOSPITAL_BASELINE:.3f} ({HOSPITAL_BASELINE:.1%})",
    'Improvement': f"{improvement:+.3f} ({improvement:+.1%})",
    'Status': 'SUCCESS - Beat Baseline' if accuracy > HOSPITAL_BASELINE else 'BELOW BASELINE'
}

if auc_score:
    summary['AUC Score'] = f"{auc_score:.3f}"

# Print summary
for key, value in summary.items():
    print(f"{key}: {value}")

# Save detailed results
try:
    with open('../final_results.txt', 'w') as f:
        f.write("HEART FAILURE PREDICTION - FINAL RESULTS\n")
        f.write("="*50 + "\n\n")
        
        for key, value in summary.items():
            f.write(f"{key}: {value}\n")
        
        f.write(f"\nConfusion Matrix:\n{cm}\n")
        f.write(f"\nClinical Metrics:\n")
        f.write(f"Sensitivity: {sensitivity:.3f}\n")
        f.write(f"Specificity: {specificity:.3f}\n")
        
        if feature_df is not None:
            f.write(f"\nTop 5 Features:\n")
            for _, row in feature_df.head(5).iterrows():
                f.write(f"  {row['Feature']}: {row['Importance']:.3f}\n")
    
    print(f"\nDetailed results saved to 'final_results.txt'")
except Exception as e:
    print(f"Error saving results: {e}")

# Project completion status
print("\n" + "="*60)
print("PROJECT COMPLETION STATUS")
print("="*60)

if accuracy > HOSPITAL_BASELINE:
    print("🎉 SUCCESS! Model beats hospital baseline")
    print(f"   Achievement: {accuracy:.1%} vs {HOSPITAL_BASELINE:.1%} baseline")
    print(f"   Improvement: {(accuracy - HOSPITAL_BASELINE) * 100:+.1f} percentage points")
else:
    print("⚠️  Model below baseline, but project complete")
    print(f"   Result: {accuracy:.1%} vs {HOSPITAL_BASELINE:.1%} baseline")
    print(f"   Gap: {(accuracy - HOSPITAL_BASELINE) * 100:.1f} percentage points")

print("\n📋 DELIVERABLES COMPLETED:")
deliverables = [
    "✓ Working machine learning model",
    "✓ Comprehensive evaluation with ROC curve", 
    "✓ Confusion matrix and clinical metrics",
    "✓ Feature importance analysis",
    "✓ Performance comparison to hospital baseline",
    "✓ Final results documentation",
    "✓ Professional visualizations and dashboard"
]

for deliverable in deliverables:
    print(f"   {deliverable}")

print("\n🏁 3-WEEK PROJECT COMPLETED SUCCESSFULLY!")